# Zero R1 OOF scores?

Global fraud CDFs, then per alarm split:

- **R1** = all three `F <= 0.1` (origin cube)
- **R2** = `y_prob <= max y_prob in R1` (every row the current model already ranks at or below the highest-scoring R1 point)

Intervention: set `y_prob = 0` on R1, leave every other score unchanged. Metric is ROC AUC (within the split, and pooled OOF).

R1 ⊆ R2 by construction. If R2 is almost the whole split, some R1 row (often one of the few R1 frauds) already has a high score, and sending R1 to 0 drags those frauds to the bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
from ccf_ml import file_utils
from ccf_ml.cross_validation import alarm as alarm_mod
from ccf_ml.features import fraud_ecdf as fraud_ecdf_mod
importlib.reload(file_utils)
importlib.reload(alarm_mod)
importlib.reload(fraud_ecdf_mod)

from ccf_ml.cross_validation.alarm import alarm_flag_mask
from ccf_ml.cross_validation.constants import (
    ALARM_FLAG_COLUMNS,
    ID_COLUMN,
    TARGET_COLUMN,
)
from ccf_ml.features.fraud_ecdf import (
    FraudEcdf,
    fraud_ecdf_from_frame,
    transform_frame,
)
from ccf_ml.file_utils import (
    get_script_table_dir,
    get_train_csv_path,
    get_training_oof_path,
)
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

DELIMITER_VALUE = "-" * 50
NOTEBOOK_STEM = "r1_zero_oof_auc"
EXP_NAME = "hgb_baseline"
PROB_COL = "y_prob"
FOLD_COL = "fold_id"
FRAUD_ECDF_COLUMNS = (
    "customer_age",
    "merchant_risk_score",
    "velocity_score",
)
ALARM_SPLIT_COL = "alarm_flag_split"
ALARM_SPLIT_LABELS = {
    False: "no alarm",
    True: "at least one alarm",
}
ALARM_SPLIT_ORDER = ["no alarm", "at least one alarm"]
LOW_CUBE = 0.1
tables_dir = get_script_table_dir(NOTEBOOK_STEM)

train_df = pd.read_csv(get_train_csv_path())
oof_df = pd.read_csv(get_training_oof_path(EXP_NAME))
train_features = train_df.drop(columns=[TARGET_COLUMN])
frame = train_features.merge(
    oof_df,
    on=ID_COLUMN,
    how="inner",
    validate="one_to_one",
)
if len(frame) != len(train_df) or len(frame) != len(oof_df):
    raise ValueError(
        f"join lost rows: train={len(train_df)} oof={len(oof_df)} joined={len(frame)}"
    )
has_alarm = alarm_flag_mask(frame, columns=ALARM_FLAG_COLUMNS)
frame[ALARM_SPLIT_COL] = has_alarm.map(ALARM_SPLIT_LABELS)
n_columns = len(FRAUD_ECDF_COLUMNS)
curve_slots: list[FraudEcdf | None] = [None] * n_columns
for i in range(n_columns):
    name = FRAUD_ECDF_COLUMNS[i]
    curve_slots[i] = fraud_ecdf_from_frame(frame, column=name)
curves = tuple(curve for curve in curve_slots if curve is not None)
if len(curves) != n_columns:
    raise ValueError("failed to estimate every fraud CDF")
cube = transform_frame(frame, curves=curves)
print("exp_name", EXP_NAME)
print("n_rows", len(cube))


## R1 vs R2, and AUC after setting R1 to 0

In [ ]:
def auc(y: np.ndarray, prob: np.ndarray) -> float:
    return float(roc_auc_score(y, prob))


def region_row(
    part: pd.DataFrame,
    *,
    region: str,
    mask: np.ndarray,
) -> dict[str, object]:
    n = int(np.sum(mask))
    y = part[TARGET_COLUMN].to_numpy()
    n_fraud = int(np.sum(mask & (y == 1)))
    n_part = len(part)
    n_part_fraud = int(np.sum(y == 1))
    return {
        "region": region,
        "n": n,
        "n_fraud": n_fraud,
        "rate": n_fraud / n if n else float("nan"),
        "share_of_rows": n / n_part if n_part else float("nan"),
        "share_of_fraud": (
            n_fraud / n_part_fraud if n_part_fraud else float("nan")
        ),
    }


y_all = cube[TARGET_COLUMN].to_numpy()
p_all = cube[PROB_COL].to_numpy(dtype=np.float64)
pooled_base = auc(y_all, p_all)
print("pooled OOF AUC", round(pooled_base, 6))

n_splits = len(ALARM_SPLIT_ORDER)
n_region_kinds = 2
region_rows: list[dict[str, object] | None] = [None] * (n_splits * n_region_kinds)
auc_rows: list[dict[str, object] | None] = [None] * n_splits
r1_masks: list[np.ndarray | None] = [None] * n_splits
region_i = 0
for split_i in range(n_splits):
    name = ALARM_SPLIT_ORDER[split_i]
    split_mask = cube[ALARM_SPLIT_COL].to_numpy() == name
    part = cube.loc[split_mask]
    vals = part.loc[:, list(FRAUD_ECDF_COLUMNS)].to_numpy(dtype=np.float64)
    r1 = np.all(vals <= LOW_CUBE, axis=1)
    if not np.any(r1):
        raise ValueError(f"R1 is empty in {name}")
    max_r1 = float(part.loc[r1, PROB_COL].max())
    r2 = part[PROB_COL].to_numpy(dtype=np.float64) <= max_r1
    r1_full = np.zeros(len(cube), dtype=bool)
    r1_full[np.flatnonzero(split_mask)] = r1
    r1_masks[split_i] = r1_full
    y = part[TARGET_COLUMN].to_numpy()
    p = part[PROB_COL].to_numpy(dtype=np.float64)
    p_zero = p.copy()
    p_zero[r1] = 0.0
    r1_fraud_scores = part.loc[r1 & (part[TARGET_COLUMN] == 1), PROB_COL].to_numpy(
        dtype=np.float64
    )
    r1_non_scores = part.loc[r1 & (part[TARGET_COLUMN] == 0), PROB_COL].to_numpy(
        dtype=np.float64
    )
    print(DELIMITER_VALUE)
    print(name)
    print("max y_prob in R1", round(max_r1, 6))
    print("R1 fraud y_prob", np.round(np.sort(r1_fraud_scores), 6))
    print(
        "R1 non-fraud y_prob min/median/max",
        round(float(np.min(r1_non_scores)), 6),
        round(float(np.median(r1_non_scores)), 6),
        round(float(np.max(r1_non_scores)), 6),
    )
    base = auc(y, p)
    zeroed = auc(y, p_zero)
    print("within-split AUC", round(base, 6), "->", round(zeroed, 6), "delta", round(zeroed - base, 6))
    region_rows[region_i] = region_row(part, region="R1 all F <= 0.1", mask=r1)
    region_rows[region_i][ALARM_SPLIT_COL] = name
    region_rows[region_i]["max_y_prob"] = max_r1
    region_i += 1
    region_rows[region_i] = region_row(
        part,
        region="R2 y_prob <= max y_prob(R1)",
        mask=r2,
    )
    region_rows[region_i][ALARM_SPLIT_COL] = name
    region_rows[region_i]["max_y_prob"] = max_r1
    region_i += 1
    auc_rows[split_i] = {
        ALARM_SPLIT_COL: name,
        "n_r1": int(np.sum(r1)),
        "n_r1_fraud": int(np.sum(r1 & (y == 1))),
        "n_r2": int(np.sum(r2)),
        "n_r2_fraud": int(np.sum(r2 & (y == 1))),
        "max_y_prob_r1": max_r1,
        "auc": base,
        "auc_r1_to_0": zeroed,
        "delta": zeroed - base,
    }

regions = pd.DataFrame([row for row in region_rows if row is not None])
auc_table = pd.DataFrame([row for row in auc_rows if row is not None])
print(DELIMITER_VALUE)
print(regions.round(6))
print(DELIMITER_VALUE)
print(auc_table.round(6))

filled_r1 = [mask for mask in r1_masks if mask is not None]
if len(filled_r1) != n_splits:
    raise ValueError("missing an R1 mask")
pooled_rows: list[dict[str, object] | None] = [None] * (n_splits + 2)
pooled_rows[0] = {
    "rule": "original",
    "auc": pooled_base,
    "delta": 0.0,
}
for split_i in range(n_splits):
    name = ALARM_SPLIT_ORDER[split_i]
    p_mod = p_all.copy()
    p_mod[filled_r1[split_i]] = 0.0
    score = auc(y_all, p_mod)
    pooled_rows[split_i + 1] = {
        "rule": f"R1 to 0 in {name} only",
        "auc": score,
        "delta": score - pooled_base,
    }
both = filled_r1[0] | filled_r1[1]
p_both = p_all.copy()
p_both[both] = 0.0
score_both = auc(y_all, p_both)
pooled_rows[n_splits + 1] = {
    "rule": "R1 to 0 in both splits",
    "auc": score_both,
    "delta": score_both - pooled_base,
}
pooled = pd.DataFrame([row for row in pooled_rows if row is not None])
print(DELIMITER_VALUE)
print("pooled OOF")
print(pooled.round(6))

regions.to_csv(tables_dir / "r1_r2_occupancy.csv", index=False)
auc_table.to_csv(tables_dir / "within_split_auc.csv", index=False)
pooled.to_csv(tables_dir / "pooled_auc.csv", index=False)
print("saved", tables_dir)
